In [19]:
# loading the libraries
import pandas as pd
import unicodedata

## Data Loading And Understanding

In [20]:
# Loading the data
df = pd.read_csv(r"data\no_show.csv")

# previewing the first five rows
df.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [21]:
print(f"The dataset has {df.shape[0]} rows and {df.shape[1]} columns")

The dataset has 110527 rows and 14 columns


In [22]:
# Checking info about the data
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  str    
 3   ScheduledDay    110527 non-null  str    
 4   AppointmentDay  110527 non-null  str    
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  str    
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  str    
dtypes: float64(1), int64(8), str(5)
memory usage: 11.8 MB


In [23]:
# Summary ststistics
df.describe().T

,count,mean,std,min,25%,50%,75%,max
PatientId,110527.0,1.474963e+14,2.560949e+14,3.921784e+04,4.172614e+12,3.173184e+13,9.439172e+13,9.999816e+14
AppointmentID,110527.0,5.675305e+06,7.129575e+04,5.030230e+06,5.640286e+06,5.680573e+06,5.725524e+06,5.790484e+06
Age,110527.0,3.708887e+01,2.311020e+01,-1.000000e+00,1.800000e+01,3.700000e+01,5.500000e+01,1.150000e+02
Scholarship,110527.0,9.826558e-02,2.976748e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
Hipertension,110527.0,1.972459e-01,3.979213e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
Diabetes,110527.0,7.186479e-02,2.582651e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
Alcoholism,110527.0,3.039981e-02,1.716856e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
Handcap,110527.0,2.224796e-02,1.615427e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.000000e+00
SMS_received,110527.0,3.210256e-01,4.668727e-01,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00


## Data Cleaning

In [24]:
# Fixing column names and renaming
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace('-', '_')
)

df = df.rename(columns={
    'hipertension': 'hypertension',
    'handcap': 'handicap'
})

df.columns

Index(['patientid', 'appointmentid', 'gender', 'scheduledday',
       'appointmentday', 'age', 'neighbourhood', 'scholarship', 'hypertension',
       'diabetes', 'alcoholism', 'handicap', 'sms_received', 'no_show'],
      dtype='str')

In [25]:
# Converting dates
df['scheduledday'] = pd.to_datetime(df['scheduledday'])
df['appointmentday'] = pd.to_datetime(df['appointmentday'])

# Convert target variable
df['no_show'] = df['no_show'].map({'Yes': 1, 'No': 0})

# Convert categorical columns
categorical_cols = ['gender', 'neighbourhood']
df[categorical_cols] = df[categorical_cols].astype('category')

In [26]:
# Handle Data Quality Issues

# Removing invalid negative age values
df = df[df['age'] >= 0]

# Converting patientid to int
df['patientid'] = df['patientid'].astype('int64')

In [27]:
def clean_neighbourhood(df, top_n=15, create_risk=True):
    """
    Cleans and engineers neighbourhood-related features
    
    Parameters:
    df (DataFrame): input dataframe
    top_n (int): number of top neighbourhoods to keep
    create_risk (bool): whether to create risk segmentation
    
    Returns:
    DataFrame with new columns:
    - neighbourhood_clean
    - neighbourhood_grouped
    - neighbourhood_risk (optional)
    """
    
    df = df.copy()
    
    # Removing accents 
    def remove_accents(text):
        return unicodedata.normalize('NFKD', text)\
            .encode('ascii', errors='ignore')\
            .decode('utf-8')
    
    df['neighbourhood'] = df['neighbourhood'].astype(str).apply(remove_accents)
    
    # Fixing special characters
    df['neighbourhood'] = df['neighbourhood'].str.replace('´', "'", regex=False)
    
    # Standardizing text
    df['neighbourhood'] = (
        df['neighbourhood']
        .str.lower()
        .str.strip()
    )
    
    # Grouping top neighbourhoods
    top_neighbourhoods = df['neighbourhood'].value_counts().nlargest(top_n).index
    
    df['neighbourhood_grouped'] = df['neighbourhood'].apply(
        lambda x: x if x in top_neighbourhoods else 'other'
    )
    
    # Risk segmentation
    if create_risk:
        risk = df.groupby('neighbourhood')['no_show'].mean()
        
        high_risk = risk[risk > 0.3].index
        low_risk = risk[risk < 0.2].index
        
        def classify_area(x):
            if x in high_risk:
                return 'high_risk'
            elif x in low_risk:
                return 'low_risk'
            else:
                return 'medium_risk'
        
        df['neighbourhood_risk'] = df['neighbourhood'].apply(classify_area)
    
    return df

df = clean_neighbourhood(df)

In [28]:
df.head()

,patientid,appointmentid,gender,scheduledday,appointmentday,age,neighbourhood,scholarship,hypertension,diabetes,alcoholism,handicap,sms_received,no_show,neighbourhood_grouped,neighbourhood_risk
0,29872499824296,5642903,F,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,jardim da penha,0,1,0,0,0,0,0,jardim da penha,low_risk
1,558997776694438,5642503,M,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,jardim da penha,0,0,0,0,0,0,0,jardim da penha,low_risk
2,4262962299951,5642549,F,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,mata da praia,0,0,0,0,0,0,0,other,low_risk
3,867951213174,5642828,F,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,pontal de camburi,0,0,0,0,0,0,0,other,low_risk
4,8841186448183,5642494,F,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,jardim da penha,0,1,1,0,0,0,0,jardim da penha,low_risk


## Feature Engineering

In [29]:
# Creating number of waiting days
df['waiting_days'] = (df['appointmentday'] - df['scheduledday']).dt.days

# Removing invalid waiting days
df = df[df['waiting_days'] >= 0]

In [30]:
# Extract weekday features
df['appointment_weekday'] = df['appointmentday'].dt.day_name()
df['scheduled_weekday'] = df['scheduledday'].dt.day_name()

In [31]:
# Creating age groups
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 12, 18, 35, 60, 115],
    labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior'],
    include_lowest=True
)

In [36]:
# Checking missing values
df.isnull().sum().sum()

np.int64(0)

In [33]:
# Check for duplicates
df.duplicated().sum()

np.int64(0)

In [34]:
# Dropping unuseful column
df = df.drop(columns=['patientid', 'appointmentid'])

In [35]:
df.to_csv(r"data\no_show_cleaned.csv", index=False)